# 🥉 Bronze Ingestion Framework
## Overview
This notebook implements a **parameterized, reusable ingestion framework** for the CommerceLake project.
It dynamically ingests raw source files from Databricks Volumes into Bronze Delta tables using **Auto Loader (cloudFiles)**, preserving data as-is with no transformations.

## Parameter
| Parameter | Description | Example |
|---|---|---|
| `domain` | Domain name used to fetch the corresponding config file | `customers`, `orders`, `products` |

## Project Details
| | |
|---|---|
| **Project** | CommerceLake ETL Pipeline |
| **Layer** | Bronze (Raw Ingestion) |
| **Author** | Sandesh M S |
| **Last Updated** | 08-03-2026 |

## Architecture
```
Config File (.conf)
      ↓
Auto Loader (cloudFiles)
      ↓
Bronze Delta Table
      ↓
Audit Log
```

In [0]:
dbutils.widgets.text("domain", "")
domain = dbutils.widgets.get("domain")
print(f"Running ingestion for domain: {domain}")

%md
## ⚙️ Configuration Loading

### What are Config Files?
Configuration files are **non-executable, key-value pair files** parsed at runtime to dynamically control script behavior without hardcoding values.

### Why Config Files in CommerceLake?
Since we have **9 different domains**, each domain has its own `.conf` file containing ingestion parameters. This enables the same notebook to ingest any domain by just changing the `domain` parameter.

### Config File Location
```
/Workspace/Users/sandeshms71@gmail.com/CommerceLake/configs/{domain}.conf
```

### Config File Structure Example
| Key | Description | Example Value |
|---|---|---|
| `domain` | Domain identifier | `customers` |
| `source_path` | Raw file location in Volume | `/Volumes/main/dev_storage/commercelake_raw/customers` |
| `target_table` | Bronze Delta table name | `main.dev_bronze.bronze_customers` |
| `file_format` | Source file format | `csv` |
| `header` | Has header row? | `true` |
| `delimiter` | Column separator | `,` |
| `schema_evolution` | Schema change handling | `addNewColumns` |
| `schema_location` | Schema checkpoint path | `/Volumes/.../schema/customers` |
| `output_mode` | Write mode | `append` |
| `checkpoint_location` | Auto Loader checkpoint | `/Volumes/.../checkpoint/customers` |

In [0]:
config_path = f"/Workspace/Users/sandeshms71@gmail.com/CommerceLake/configs/{domain}.conf"

config = {}

try:
    with open(config_path, "r") as f:
        for line in f:
            key, value = line.strip().split("=",1)
            config[key.strip()] = value.strip()
        print(config)
except FileNotFoundError:
    raise Exception(f"Config file not found for domain: {domain}")

required_keys = ['source_path', 'target_table', 'file_format','checkpoint_location','schema_evolution','schema_location']
for key in required_keys:
    if key not in config:
        raise Exception(f"Missing required config key: {key}")

## 🔄 Data Ingestion — Auto Loader (cloudFiles)

### What is Auto Loader?
Auto Loader is a highly efficient, scalable tool for incrementally ingesting new data files from cloud storage (S3, ADLS, GCS) into Delta Lake as they arrive.
It uses structured streaming (`cloudFiles` source) to automatically discover files, manage schema evolution, and handle millions of files supporting formats like CSV, JSON, Parquet and Avro.

### Why Auto Loader for CommerceLake?
| Feature | Benefit |
|---|---|
| **Incremental processing** | Only new files processed each run |
| **Schema evolution** | Handles new columns automatically |
| **Scalable** | Handles millions of files efficiently |
| **Fault tolerant** | Checkpoint ensures no data loss on restart |

### Audit Columns Added
Every ingested record gets 3 additional audit columns:

| Column | Description | Source |
|---|---|---|
| `ingestion_date` | Date when file was ingested | `current_date()` |
| `file_modify_time` | Last modified time of source file | `_metadata.file_modification_time` |
| `file_path` | Full path of source file | `_metadata.file_path` |

In [0]:
%sql
CREATE TABLE IF NOT EXISTS main.dev_bronze.bronze_ingestion_audit (
    domain STRING,
    target_table STRING,
    rows_written BIGINT,
    run_timestamp TIMESTAMP,
    run_status STRING
)
USING DELTA


In [0]:
from pyspark.sql.functions import *
row_count = 0
run_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
run_status = 'FAILED'

try:
    df=spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format",config['file_format'])\
        .option("header", config['header'])\
        .option("delimiter", config['delimiter'])\
        .option("cloudFiles.schemaEvolutionMode",config['schema_evolution'])\
        .option("cloudFiles.schemaLocation",config['schema_location'])\
        .load(config['source_path'])

    df = df.withColumn('ingestion_date', current_date())\
        .withColumn('file_modify_time', col("_metadata.file_modification_time"))\
        .withColumn('file_path', col("_metadata.file_path"))

    df.writeStream.format("delta")\
        .outputMode(config['output_mode'])\
        .option("checkpointLocation",config['checkpoint_location'])\
        .trigger(availableNow = True)\
        .toTable(config['target_table'])

    row_count = spark.read.table(config['target_table']).count()
    run_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
    run_status = 'SUCCESS'
except Exception as e:
    run_status='FAILED'
    raise Exception(f"Ingestion failed for domain '{domain}': {str(e)}")

finally:
    audit_data = [(domain, config['target_table'], row_count, run_time,run_status)]
    audit_df = spark.createDataFrame(audit_data,["domain", "target_table", "rows_written", "run_timestamp", "run_status"])
    audit_df.write.mode("append").saveAsTable("main.dev_bronze.bronze_ingestion_audit")


%md
## 📊 Audit Logging

### What is the Audit Table?
A Delta table that tracks the execution history of every bronze ingestion run, providing visibility into data pipeline health and row-level statistics.

### Why Audit Logging?
| Benefit | Detail |
|---|---|
| **Observability** | Track how many records were written per run |
| **Debugging** | Identify failed or empty runs quickly |
| **Lineage** | Know exactly when data arrived |
| **Idempotency** | Verify no duplicate or missing runs |

### Audit Table Schema
| Column | Type | Description | Example |
|---|---|---|---|
| `domain` | STRING | Source domain name | `customers` |
| `target_table` | STRING | Bronze table written to | `main.dev_bronze.bronze_customers` |
| `rows_written` | BIGINT | Total rows in table after run | `99441` |
| `run_timestamp` | TIMESTAMP | Execution time | `2026-03-08 03:05:45` |
| `run_status` | STRING | Run result | `SUCCESS` |

### Audit Table Location
```
main.dev_bronze.bronze_ingestion_audit
```

```
